# Expérimentations préliminaires — 2. XGBoost

Gradient boosting d'arbres de décision : modèle non linéaire capable de représenter des
interactions entre variables.

Pipeline : imputation (médiane) → standardisation → **ACP (critère de Kaiser, λ > 1)** →
XGBoost, comme pour la régression logistique. Les arbres n'ont besoin ni de standardisation ni
d'ACP pour fonctionner ; l'ACP sert ici uniquement à **alléger le modèle** : elle remplace les
1 060 variables, très redondantes, par un nombre beaucoup plus petit de composantes, ce qui
accélère l'entraînement et réduit le nombre de divisions candidates, donc le risque de
surapprentissage sur du bruit. Pour le choix des hyperparamètres, le prétraitement est ajusté
sur les données d'ajustement seulement ; il est ensuite réajusté sur tout l'entraînement du
pli pour le modèle final. Rien n'est ajusté sur la saison testée.

Réglage volontairement simple (étude préliminaire) :

- taux d'apprentissage 0,03, sous-échantillonnage des matchs et des variables à 80 % ;
- profondeur maximale choisie dans {2, 3, 4} sur la validation interne ;
- nombre d'arbres fixé par arrêt précoce (100 itérations sans amélioration de la log loss de
  validation), puis réajustement sur tout l'entraînement du pli avec ce nombre d'arbres.

**Protocole commun** (voir `src/nhl_forecast/features/equipes.py` et
`src/nhl_forecast/evaluation/preliminaire.py`) :

- saisons régulières **2009-10 à 2025-26**, statistiques d'**équipes** seulement, **toutes les
  colonnes** de `t_summary`, `t_percentages`, `t_summaryshooting`, `t_outshootoutshotby` et
  MoneyPuck (5 situations de jeu) ; seuls les identifiants, les colonnes vides ou constantes et
  les doublons exacts sont retirés, automatiquement ;
- validation **progressive** (*walk-forward*, fenêtre croissante) : chaque pli teste **une
  saison** et s'entraîne sur toutes les saisons antérieures ;
  - **développement** : plis 1 à 8, saisons testées 2016-17 à 2023-24 ;
  - **évaluation finale** : plis 9 et 10, saisons 2024-25 et 2025-26, réservées (à ne pas
    utiliser pour les choix de modélisation) ;
- hyperparamètres choisis sur les 20 % les plus récents des matchs d'entraînement du pli ;
- métriques : log loss, score de Brier, accuracy (seuil 0,5), résumées séparément pour chaque
  phase.

Écart avec le protocole proposé (initialisation 2005-06 à 2007-08, développement à partir de
2008-09) : MoneyPuck ne commence qu'en 2008-09, et `t_percentages` / `t_summaryshooting` en
2009-10. Les saisons 2005-06 à 2008-09 sont donc exclues de ce jeu de statistiques d'équipes ;
elles pourront servir à l'initialisation des modèles qui n'utilisent que les résultats (Elo).

Avec `SAUVER = True`, les tableaux sont enregistrés en CSV (séparateur « ; ») dans
`reports/map6009/tables/` (préfixe `exp_`) et les prédictions dans
`data/processed/experimentation/` (lues par `04_comparaison.ipynb`).

L'ancien protocole (2020-21 à 2025-26, une saison testée par pli ; rapport préliminaire 1) reste
disponible en définissant `NHL_EXP=base` avant les imports ; ses sorties sont suffixées `_base`
(`tables_base/`, `figures_base/`, `experimentation_base/`).

In [1]:
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

from nhl_forecast.evaluation.preliminaire import (
    GRAINE,
    ACPKaiser,
    exporter_tableau,
    formater,
    plis_chronologiques,
    predictions,
    resultats_par_pli,
    resume,
    sauver_predictions,
    separation_interne,
)
from nhl_forecast.features.equipes import NOM_SAISON, charger_equipes, construire_matchs, variables

SAUVER = True
PROFONDEURS = [2, 3, 4]
PARAMS = dict(objective="binary:logistic", eval_metric="logloss", learning_rate=0.03,
              subsample=0.8, colsample_bytree=0.8, min_child_weight=5, reg_lambda=1.0,
              tree_method="hist", random_state=GRAINE, n_jobs=-1)

In [2]:
matchs = construire_matchs(charger_equipes())
X_cols = variables(matchs)
X, y = matchs[X_cols], matchs["victoire_locale"].to_numpy()
plis = plis_chronologiques(matchs)
print(f"{len(matchs)} matchs, {len(X_cols)} variables")

20382 matchs, 1060 variables


C:\Users\tom_9\Projets\nhl-probabilistic-forecasting\src\nhl_forecast\features\equipes.py:205: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  matchs = matchs.astype({"visiteur": "int64"}).reset_index()


## 1. Entraînement par pli

In [3]:
def pretraitement():
    return make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                         ACPKaiser(random_state=GRAINE))  # fmt: skip


preds, choix, acp, importances = [], [], [], []
for pli in plis:
    # (a) profondeur et nombre d'arbres : ACP ajustée sur l'ajustement seulement
    ajust, val = separation_interne(pli["entr"])
    pre = pretraitement().fit(X.iloc[ajust])
    Z_ajust, Z_val = pre.transform(X.iloc[ajust]), pre.transform(X.iloc[val])
    essais = {}
    for d in PROFONDEURS:
        m = XGBClassifier(**PARAMS, max_depth=d, n_estimators=3000, early_stopping_rounds=100)
        m.fit(Z_ajust, y[ajust], eval_set=[(Z_val, y[val])], verbose=False)
        essais[d] = (m.best_score, m.best_iteration + 1)
    d = min(essais, key=lambda k: essais[k][0])
    perte, n_arbres = essais[d]
    # (b) modèle final : ACP réajustée sur tout l'entraînement du pli
    pre = pretraitement().fit(X.iloc[pli["entr"]])
    m = XGBClassifier(**PARAMS, max_depth=d, n_estimators=n_arbres)
    m.fit(pre.transform(X.iloc[pli["entr"]]), y[pli["entr"]])
    p = m.predict_proba(pre.transform(X.iloc[pli["test"]]))[:, 1]
    preds.append(predictions(matchs, pli, p, "XGBoost"))
    saison = NOM_SAISON[pli["saison_test"]]
    choix.append({"Pli": pli["pli"], "Saison testée": saison,
                  "Profondeur": d, "Arbres": n_arbres, "Log loss (validation)": perte})
    acp.append({"Pli": pli["pli"], "Saison testée": saison,
                "Composantes ACP": pre[-1].n_components_,
                "Variance expliquée (%)": round(100 * pre[-1].variance_expliquee_, 1)})
    gain = m.get_booster().get_score(importance_type="gain")  # clés f0, f1, … = CP1, CP2, …
    importances.append(pd.Series({f"CP{int(k[1:]) + 1}": v for k, v in gain.items()},
                                 name=pli["pli"]))
preds = pd.concat(preds, ignore_index=True)
tab_choix = pd.DataFrame(choix)
tab_acp = pd.DataFrame(acp)
formater(tab_choix)

Pli,Saison testée,Profondeur,Arbres,Log loss (validation)
1,2016-17,4,145,0.6834
2,2017-18,2,66,0.6796
3,2018-19,2,411,0.6775
4,2019-20,3,178,0.6770
5,2020-21,4,146,0.6814
6,2021-22,3,170,0.6780
7,2022-23,2,189,0.6718
8,2023-24,2,207,0.6680
9,2024-25,2,243,0.6640
10,2025-26,2,190,0.6669


In [4]:
formater(tab_acp)

Pli,Saison testée,Composantes ACP,Variance expliquée (%)
1,2016-17,152,90.6000
2,2017-18,153,90.4000
3,2018-19,155,90.3000
4,2019-20,156,90.2000
5,2020-21,157,90.2000
6,2021-22,158,90.2000
7,2022-23,159,90.2000
8,2023-24,158,90.1000
9,2024-25,159,90.1000
10,2025-26,159,90.0000


## 2. Résultats par pli

In [5]:
tab_plis = resultats_par_pli(preds)
if SAUVER:
    exporter_tableau(tab_plis, "exp_xgboost_par_pli")
    exporter_tableau(tab_choix, "exp_xgboost_hyperparametres")
    exporter_tableau(tab_acp, "exp_xgboost_composantes")
    sauver_predictions(preds, "02_xgboost")
formater(tab_plis)

Modèle,Pli,Phase,Saison testée,n,Log loss,Brier,Accuracy
XGBoost,1,Développement,2016-17,1230,0.6801,0.2436,0.5593
XGBoost,2,Développement,2017-18,1271,0.6811,0.2440,0.5956
XGBoost,3,Développement,2018-19,1271,0.6830,0.2449,0.5728
XGBoost,4,Développement,2019-20,1082,0.6847,0.2459,0.5434
XGBoost,5,Développement,2020-21,868,0.6718,0.2396,0.5749
XGBoost,6,Développement,2021-22,1312,0.6627,0.2350,0.6075
XGBoost,7,Développement,2022-23,1312,0.6686,0.2379,0.6006
XGBoost,8,Développement,2023-24,1312,0.6676,0.2375,0.5991
XGBoost,9,Évaluation finale,2024-25,1312,0.6682,0.2378,0.5869
XGBoost,10,Évaluation finale,2025-26,1312,0.6844,0.2457,0.5480


## 3. Résumé (moyenne ± écart-type sur les saisons testées, par phase)

In [6]:
tab_resume = resume(preds)
if SAUVER:
    exporter_tableau(tab_resume, "exp_xgboost_resume")
formater(tab_resume)

Modèle,Phase,Saisons,Log loss,Brier,Accuracy
XGBoost,Développement,8,0.6749 ± 0.0083,0.2411 ± 0.0040,0.5817 ± 0.0227
XGBoost,Évaluation finale,2,0.6763 ± 0.0115,0.2418 ± 0.0056,0.5675 ± 0.0275


## 4. Importance des composantes

Gain moyen apporté par les divisions sur chaque composante principale, normalisé en % du gain
total de chaque pli, puis moyenné sur les 10 plis (0 si la composante n'existe pas ou n'est
pas utilisée dans un pli). Les composantes sont numérotées par valeur propre décroissante ;
l'ACP étant réajustée à chaque pli, une même composante (p. ex. CP1) n'a pas exactement la
même définition d'un pli à l'autre.

In [7]:
imp = pd.concat(importances, axis=1).fillna(0)
imp = 100 * imp / imp.sum()
tab_imp = (pd.DataFrame({"Gain moyen (%)": imp.mean(axis=1), "Écart-type": imp.std(axis=1)})
           .sort_values("Gain moyen (%)", ascending=False).head(15)
           .rename_axis("Composante").reset_index())
if SAUVER:
    exporter_tableau(tab_imp, "exp_xgboost_importance", 2)
formater(tab_imp, 2)

Composante,Gain moyen (%),Écart-type
CP1,2.53,1.14
CP13,1.02,0.38
CP6,0.94,0.42
CP9,0.91,0.44
CP76,0.91,0.40
CP8,0.90,0.35
CP111,0.89,0.41
CP10,0.88,0.42
CP79,0.88,0.40
CP14,0.88,0.33
